## Resources

In [1]:
import pandas as pd
import numpy as np
import re
import gc
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install catboost
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
from xgboost import XGBClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.8 MB/s eta 0:00:00


# 1. SETUP & CONFIGURATION

In [2]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }

}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

Local file for train not found. Fetching from GitHub...
Loaded train from GitHub successfully.
Local file for test not found. Fetching from GitHub...
Loaded test from GitHub successfully.
Local file for sample_sub not found. Fetching from GitHub...
Loaded sample_sub from GitHub successfully.


In [3]:
SEED = 42
N_SEEDS = 2
N_FOLDS = 5
eps = 1e-15

id_col = 'Tour_ID'
target_col = 'cost_category'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
NUM_CLASSES = len(target_classes)

class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

# 2. FEATURE ENGINEERING & PREPROCESSING

In [ ]:
def preprocess_dataset(df):
    df = df.copy()

    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})

    df['travel_with'] = df['travel_with'].fillna('Alone')
    df['total_female'] = df['total_female'].fillna(0)
    df['total_male'] = df['total_male'].fillna(0)
    df['most_impressing'] = df.get('most_impressing', pd.Series()).fillna('No Answer')

    df['total_people'] = df['total_female'] + df['total_male']
    df['total_people_safe'] = df['total_people'].apply(lambda x: 1 if x == 0 else x)

    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    df['total_nights_safe'] = df['total_nights'].apply(lambda x: 1 if x == 0 else x)

    df['mainland_ratio'] = df['night_mainland'] / df['total_nights_safe']
    df['zanzibar_ratio'] = df['night_zanzibar'] / df['total_nights_safe']
    df['female_ratio'] = df['total_female'] / df['total_people_safe']
    df['nights_per_person'] = df['total_nights'] / df['total_people_safe']

    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    df['package_depth'] = df['package_count'] / len(package_cols)
    df['is_full_package'] = (df['package_count'] == len(package_cols)).astype(int)
    df['has_no_package'] = (df['package_count'] == 0).astype(int)

    return df

train_df = preprocess_dataset(train)
test_df = preprocess_dataset(test)

cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity',
    'info_source', 'tour_arrangement', 'package_transport_int',
    'package_accomodation', 'package_food', 'package_transport_tz',
    'package_sightseeing', 'package_guided_tour',
    'package_insurance', 'first_trip_tz'
]

# Frequency Encoding
for col in ['country', 'purpose', 'main_activity']:
    freq_map = pd.concat([train_df[col], test_df[col]]).value_counts()
    train_df[f'{col}_freq'] = train_df[col].map(freq_map)
    test_df[f'{col}_freq'] = test_df[col].map(freq_map)

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

# Formats for CatBoost
X_train_cb = train_df[features].copy()
X_test_cb = test_df[features].copy()
for col in cat_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

# One-Hot Encoding for LightGBM and XGBoost
df_all = pd.concat([train_df[features], test_df[features]], axis=0).reset_index(drop=True)
df_all.columns = [re.sub(r'[^\w]', '_', col) for col in df_all.columns]
df_all_encoded = pd.get_dummies(df_all, columns=[re.sub(r'[^\w]', '_', col) for col in cat_features], drop_first=False)

X_train = df_all_encoded.iloc[:len(train_df)].copy()
X_test = df_all_encoded.iloc[len(train_df):].copy()
y = train_df['target'].values

# 3. MODEL BUILDERS

In [ ]:
def make_lgb(seed):
    return lgb.LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        metric="multi_logloss",
        boosting_type="gbdt",
        learning_rate=0.03,
        num_leaves=48,
        min_child_samples=25,
        feature_fraction=0.75,
        bagging_fraction=0.85,
        bagging_freq=5,
        lambda_l1=0.3,
        lambda_l2=0.6,
        max_depth=-1,
        n_estimators=3000,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

def make_cat(seed):
    return CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=3000,
        learning_rate=0.04,
        depth=6,
        l2_leaf_reg=4.0,
        random_strength=1.0,
        bagging_temperature=0.6,
        border_count=128,
        random_seed=seed,
        od_type="Iter",
        od_wait=200,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

def make_xgb(seed):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        learning_rate=0.04,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.3,
        reg_lambda=0.8,
        n_estimators=3000,
        early_stopping_rounds=150,
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
    )

# 4. CROSS-VALIDATION + MULTI-SEED BAGGING

In [ ]:
oof_lgb = np.zeros((len(X_train), NUM_CLASSES))
oof_cat = np.zeros((len(X_train), NUM_CLASSES))
oof_xgb = np.zeros((len(X_train), NUM_CLASSES))

test_lgb = np.zeros((len(X_test), NUM_CLASSES))
test_cat = np.zeros((len(X_test), NUM_CLASSES))
test_xgb = np.zeros((len(X_test), NUM_CLASSES))

print("Starting Multi-Seed Stratified K-Fold CV Training...")

for seed in range(N_SEEDS):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # ---- LightGBM ----
        lgb_model = make_lgb(SEED + seed)
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)]
        )
        oof_lgb[va_idx] += lgb_model.predict_proba(X_va) / N_SEEDS
        test_lgb += lgb_model.predict_proba(X_test) / (N_FOLDS * N_SEEDS)

        # ---- CatBoost ----
        cat_model = make_cat(SEED + seed)
        tr_pool = Pool(X_train_cb.iloc[tr_idx], y_tr, cat_features=cat_features)
        va_pool = Pool(X_train_cb.iloc[va_idx], y_va, cat_features=cat_features)
        cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        oof_cat[va_idx] += cat_model.predict_proba(va_pool) / N_SEEDS
        test_cat += cat_model.predict_proba(Pool(X_test_cb, cat_features=cat_features)) / (N_FOLDS * N_SEEDS)

        # ---- XGBoost ----
        xgb_model = make_xgb(SEED + seed)
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
        oof_xgb[va_idx] += xgb_model.predict_proba(X_va) / N_SEEDS
        test_xgb += xgb_model.predict_proba(X_test) / (N_FOLDS * N_SEEDS)

        print(f"[Seed {seed}] Fold {fold} | "
              f"LGB: {log_loss(y_va, lgb_model.predict_proba(X_va)):.5f} | "
              f"CAT: {log_loss(y_va, cat_model.predict_proba(va_pool)):.5f} | "
              f"XGB: {log_loss(y_va, xgb_model.predict_proba(X_va)):.5f}")

    gc.collect()

# 5. INDIVIDUAL FAMILY EVALUATION

In [ ]:
oof_lgb_cal = np.clip(oof_lgb, eps, 1 - eps)
oof_cat_cal = np.clip(oof_cat, eps, 1 - eps)
oof_xgb_cal = np.clip(oof_xgb, eps, 1 - eps)

print("\n==================================================")
print("OUT-OF-FOLD (OOF) BASE MODEL LOG LOSS SCORES:")
print("==================================================")
print(f" LightGBM  OOF Log Loss : {log_loss(y, oof_lgb_cal):.5f}")
print(f" CatBoost  OOF Log Loss : {log_loss(y, oof_cat_cal):.5f}")
print(f" XGBoost   OOF Log Loss : {log_loss(y, oof_xgb_cal):.5f}")

# 6. OPTIMAL WEIGHT FINDER (NELDER-MEAD)

In [ ]:
def loss_func(weights):
    w1, w2, w3 = weights
    w_sum = w1 + w2 + w3
    if w_sum <= 0:
        return 999.0
    w1, w2, w3 = w1 / w_sum, w2 / w_sum, w3 / w_sum

    blend = w1 * oof_lgb + w2 * oof_cat + w3 * oof_xgb
    blend = np.clip(blend, eps, 1 - eps)
    blend = blend / blend.sum(axis=1, keepdims=True)
    return log_loss(y, blend)

init_weights = [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0]
bounds = [(0, 1), (0, 1), (0, 1)]

res = minimize(loss_func, init_weights, method='Nelder-Mead', bounds=bounds)
w_lgb, w_cat, w_xgb = res.x / np.sum(res.x)

print("\n==================================================")
print("OPTIMAL ENSEMBLE WEIGHTS:")
print("==================================================")
print(f" LightGBM  Weight : {w_lgb:.4f}")
print(f" CatBoost  Weight : {w_cat:.4f}")
print(f" XGBoost   Weight : {w_xgb:.4f}")

# 7. FINAL BLENDING & PROBABILITY CALIBRATION

In [ ]:
oof_blend = w_lgb * oof_lgb + w_cat * oof_cat + w_xgb * oof_xgb
test_blend = w_lgb * test_lgb + w_cat * test_cat + w_xgb * test_xgb

oof_blend = np.clip(oof_blend, eps, 1 - eps)
oof_blend = oof_blend / oof_blend.sum(axis=1, keepdims=True)

test_blend = np.clip(test_blend, eps, 1 - eps)
test_blend = test_blend / test_blend.sum(axis=1, keepdims=True)

final_score = log_loss(y, oof_blend)
print("\n==================================================")
print(f"FINAL OPTIMIZED ENSEMBLE OOF LOG LOSS: {final_score:.5f}")
print("==================================================")

# 8. SUBMISSION GENERATION & VALIDATION

In [ ]:
submission = pd.DataFrame(test_blend, columns=[idx_to_class[i] for i in range(NUM_CLASSES)])
submission.insert(0, id_col, test_df[id_col])

submission = sample_sub[[id_col]].merge(submission, on=id_col, how='left')
submission.to_csv('final_ensemble_submission.csv', index=False)

print("\nPredictions saved successfully to 'final_ensemble_submission.csv'")